# 03 — Team Stats Review

**Source:** 148 columns from `team_stats_season.parquet` (74 unique metrics × 2 sides: FROM + TO)  
**Goal:** Understand what each metric measures, spot redundancy, check coverage, and decide what to keep.

---

### Key context
- `from_team_stats_*` = stats of the team the player **left** (that season)
- `to_team_stats_*` = stats of the team the player **joined** (that season)
- These are **team-level aggregates** for the entire season, NOT player-specific
- Both FROM and TO team stats are **known before** we need to predict → safe features (no leakage)

### Questions
1. What are the 74 metrics? How to group them?
2. Coverage / null rates?
3. Which metrics are highly correlated (redundant)?
4. What's the distribution of FROM vs TO? (do players move to better/worse teams?)
5. **What to keep, what to drop?**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import subprocess
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=0.85)
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 120)
pd.set_option('display.float_format', '{:.3f}'.format)

BASE = Path.home() / 'Documents'
def find_file(name):
    r = subprocess.run(['find', str(BASE), '-name', name], capture_output=True, text=True)
    lines = [l for l in r.stdout.strip().split('\n') if l]
    return Path(lines[0]) if lines else None

PARQUET = find_file('transfers_model_2018_2025.parquet')
if PARQUET is None:
    PARQUET = find_file('transfers_model_v2_2018_2025.parquet')
print(f'Loading: {PARQUET}')
df = pd.read_parquet(PARQUET)
print(f'Shape: {df.shape[0]:,} × {df.shape[1]}')

In [ ]:
# Extract team stats columns
from_ts = sorted([c for c in df.columns if c.startswith('from_team_stats_')])
to_ts = sorted([c for c in df.columns if c.startswith('to_team_stats_')])

# Unique metric names (strip prefix)
from_names = [c.replace('from_team_stats_', '') for c in from_ts]
to_names = [c.replace('to_team_stats_', '') for c in to_ts]

print(f'FROM team stats: {len(from_ts)} columns')
print(f'TO team stats:   {len(to_ts)} columns')
print(f'Unique metrics:  {len(set(from_names))} FROM, {len(set(to_names))} TO')
print(f'FROM == TO names? {set(from_names) == set(to_names)}')

## 1. The 74 metrics — what are they?

Organized by logical domain.

In [ ]:
# Proposed grouping — feel free to adjust
team_groups = {
    'Attacking output': [
        'goals', 'xg', 'np_goals', 'np_xg', 'np_xg_per_shot', 
        'shots', 'np_shots', 'shots_on_target', 'high_opportunity_shots',
        'xg_from_direct_attacks', 'xg_from_sustained_attacks',
    ],
    'Box penetration': [
        'box_entries', 'box_touches', 'num_box_entries',
        'box_entries_from_carries_pct', 'box_entries_from_crosses_pct',
        'box_to_shot_pct', 'final_third_to_box_pct',
        'num_possessions_final_third', 'possessions_to_final_third_pct',
    ],
    'Possession & style': [
        'ball_possession_pct', 'field_tilt_pct', 'pass_tempo',
        'long_ball_pct', 'forward_passes_from_middle_third_pct', 'ppda',
        'ball_in_play_minutes',
    ],
    'Pressing & recovery': [
        'defensive_intensity', 'time_to_defensive_action_s', 'time_to_recovery_s',
        'recoveries', 'recoveries_within_5s_pct', 'num_recoveries_att_half',
        'time_to_defensive_action_after_loss_own_half_s',
        'time_to_defensive_action_after_loss_att_half_s',
        'xg_within_10s_after_recovery', 'xt_within_10s_after_recovery',
        'possessions_to_box_within_10s_after_recovery',
        'possessions_to_final_third_within_10s_after_recovery',
        'final_third_entry_within_10s_after_recovery_own_half_pct',
        'box_entry_within_10_after_recovery_att_half_pct',
        'first_pass_forward_after_recovery_own_half_pct',
        'median_time_to_first_forward_pass_own_half_s',
    ],
    'Defensive solidity': [
        'defensive_action_height_m', 'recovery_line_height_m',
        'defensive_duels_won_pct',
        'possessions_retained_after_5s', 'possessions_retained_after_5s_pct',
    ],
    'Attack patterns': [
        'shots_from_direct_attacks_pct', 'shots_from_sustained_attacks_pct',
        'num_shots_from_direct_attacks', 'num_shots_from_sustained_attacks',
        'crosses_per_final_third_possession', 'dribbles_per_final_third_possession',
        'shots_per_final_third_pass', 'buildups_from_goalkicks_pct',
        'shots_from_outside_box_pct', 'final_third_recoveries_pct',
    ],
    'Set pieces & discipline': [
        'corners', 'penalties', 'fouls_commited', 'fouls_in_attacking_half_pct',
        'red_cards', 'yellow_cards', 'offsides', 'own_goals',
        'num_throwins_final_third',
    ],
    'Results & standings': [
        'points', 'goal_difference', 'xpts', 'win_probability_pct',
        'xpts_points_diff', 'match_duration_minutes', 'xt',
    ],
}

# Check all 74 are accounted for
grouped = set()
for g, metrics in team_groups.items():
    grouped.update(metrics)
ungrouped = set(from_names) - grouped

print(f'Grouped: {len(grouped)} / {len(from_names)} metrics')
if ungrouped:
    print(f'UNGROUPED ({len(ungrouped)}): {sorted(ungrouped)}')

print()
for group, metrics in team_groups.items():
    actual = [m for m in metrics if m in from_names]
    missing = [m for m in metrics if m not in from_names]
    print(f'\n--- {group} ({len(actual)} metrics) ---')
    for m in actual:
        print(f'  {m}')
    if missing:
        print(f'  [NOT FOUND: {missing}]')

## 2. Coverage / null rates

In [ ]:
# Null rates for all team stats
null_from = pd.Series({c.replace('from_team_stats_', ''): df[c].isnull().mean() * 100 for c in from_ts})
null_to = pd.Series({c.replace('to_team_stats_', ''): df[c].isnull().mean() * 100 for c in to_ts})

null_compare = pd.DataFrame({'FROM null %': null_from, 'TO null %': null_to}).sort_values('FROM null %', ascending=False)

print(f'FROM team stats — avg null: {null_from.mean():.1f}%, max: {null_from.max():.1f}%')
print(f'TO team stats   — avg null: {null_to.mean():.1f}%, max: {null_to.max():.1f}%')
print(f'\nMetrics with >10% null (FROM): {(null_from > 10).sum()}')
print(f'Metrics with >10% null (TO):   {(null_to > 10).sum()}')
print()
null_compare.style.format('{:.1f}').background_gradient(cmap='YlOrRd')

In [ ]:
# Null rate by season — is it temporal?
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, season_col, ts_cols, label in [
    (axes[0], 'from_season', from_ts, 'FROM team stats'),
    (axes[1], 'to_season', to_ts, 'TO team stats'),
]:
    seasons = sorted(df[season_col].dropna().unique())
    null_by_season = []
    for s in seasons:
        mask = df[season_col] == s
        null_by_season.append(df.loc[mask, ts_cols].isnull().mean().mean() * 100)
    ax.bar(range(len(seasons)), null_by_season, color='steelblue')
    ax.set_xticks(range(len(seasons)))
    ax.set_xticklabels(seasons, rotation=45, fontsize=8)
    ax.set_ylabel('Avg null %')
    ax.set_title(f'{label} — avg null % by season')
    for i, v in enumerate(null_by_season):
        ax.text(i, v + 0.5, f'{v:.0f}%', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

## 3. Distributions — FROM vs TO

Do players systematically move to better/worse teams? This shows the team-quality shift.

In [ ]:
# Key metrics: compare FROM vs TO distributions
key_team = ['ball_possession_pct', 'xg', 'ppda', 'defensive_intensity',
            'points', 'goal_difference', 'box_entries', 'defensive_action_height_m']
key_team = [m for m in key_team if f'from_team_stats_{m}' in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for i, metric in enumerate(key_team):
    ax = axes[i]
    f_col = f'from_team_stats_{metric}'
    t_col = f'to_team_stats_{metric}'
    
    ax.hist(df[f_col].dropna(), bins=40, alpha=0.6, color='steelblue', label='FROM team', density=True)
    ax.hist(df[t_col].dropna(), bins=40, alpha=0.6, color='tomato', label='TO team', density=True)
    ax.set_title(metric, fontsize=9)
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=7)

for j in range(len(key_team), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('FROM vs TO team stats — do players move to better or worse teams?', fontsize=11)
plt.tight_layout()
plt.show()

## 4. Correlation between metrics — finding redundancy

74 metrics is a lot. Many will be measuring similar things (e.g., `xg` and `goals` and `np_xg`).  
Let's find the most correlated pairs.

In [ ]:
# Correlation matrix for FROM team stats
corr = df[from_ts].corr()
corr.columns = from_names
corr.index = from_names

fig, ax = plt.subplots(figsize=(22, 18))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            ax=ax, square=True, xticklabels=True, yticklabels=True)
ax.tick_params(labelsize=6)
ax.set_title('FROM team stats — correlation matrix (74 metrics)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Extract highly correlated pairs (|r| > 0.85)
high_corr_pairs = []
for i in range(len(from_names)):
    for j in range(i+1, len(from_names)):
        r = corr.iloc[i, j]
        if abs(r) > 0.85:
            high_corr_pairs.append({
                'metric_a': from_names[i],
                'metric_b': from_names[j],
                'corr': r
            })

hcp = pd.DataFrame(high_corr_pairs).sort_values('corr', ascending=False, key=abs)
print(f'Pairs with |corr| > 0.85: {len(hcp)}')
print()
hcp.style.format({'corr': '{:+.3f}'}).background_gradient(cmap='RdYlGn_r', subset=['corr'], vmin=0.85, vmax=1)

## 5. How correlated are FROM and TO for the same metric?

If `from_team_stats_X` and `to_team_stats_X` are highly correlated, it means players tend to move to teams of similar quality. If not, the delta provides useful info.

In [ ]:
# FROM ↔ TO correlation for each metric
from_to_corr = []
for name, f_col, t_col in zip(from_names, from_ts, to_ts):
    mask = df[f_col].notna() & df[t_col].notna()
    if mask.sum() > 500:
        r = df.loc[mask, f_col].corr(df.loc[mask, t_col])
        from_to_corr.append({'metric': name, 'from_to_corr': r})

ftc = pd.DataFrame(from_to_corr).sort_values('from_to_corr', ascending=False)

fig, ax = plt.subplots(figsize=(10, 14))
colors = ['tomato' if r > 0.7 else 'steelblue' for r in ftc['from_to_corr']]
ax.barh(range(len(ftc)), ftc['from_to_corr'], color=colors)
ax.set_yticks(range(len(ftc)))
ax.set_yticklabels(ftc['metric'], fontsize=7)
ax.axvline(0.7, color='red', ls='--', alpha=0.3, label='r=0.7 (high similarity)')
ax.set_xlabel('Correlation (FROM ↔ TO same metric)')
ax.set_title('Do players move to similar teams?\nFROM ↔ TO correlation per team stat')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Metrics with FROM↔TO corr > 0.7: {(ftc["from_to_corr"] > 0.7).sum()} / {len(ftc)}')
print(f'Metrics with FROM↔TO corr < 0.3: {(ftc["from_to_corr"] < 0.3).sum()} / {len(ftc)}')

## 6. Summary

**What the data tells us. You decide.**

| Domain | # metrics | Likely redundancy | Notes |
|--------|-----------|-------------------|-------|
| Attacking output | ~11 | High (goals ↔ xg ↔ np_xg) | Could pick 2-3 representative |
| Box penetration | ~9 | Medium | box_entries and num_box_entries might overlap |
| Possession & style | ~7 | Low | Fairly orthogonal |
| Pressing & recovery | ~16 | High among transition metrics | Many are variations of "speed of recovery" |
| Defensive solidity | ~5 | Low | Compact already |
| Attack patterns | ~10 | Medium | direct vs sustained attack variations |
| Set pieces & discipline | ~9 | Low | Mostly counts |
| Results & standings | ~7 | Medium (points ↔ xpts) | |

### Options
1. **Keep all 74 × 2 = 148** — let the model figure out redundancy
2. **Prune highly correlated pairs** — drop one from each pair with |r| > 0.9
3. **Keep only representatives per domain** — 3-5 per domain ≈ 30-40 × 2

In [ ]:
# Quick stats
print('TEAM STATS SUMMARY')
print(f'Total team stat columns: {len(from_ts) + len(to_ts)}')
print(f'Unique metrics: {len(from_names)}')
print(f'Avg null rate (FROM): {null_from.mean():.1f}%')
print(f'Avg null rate (TO): {null_to.mean():.1f}%')
print(f'Highly correlated pairs (|r|>0.85): {len(hcp)}')